[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philmui/worldmodels/blob/main/gait/skeleton-jepa/gavd/03-build-pretraining-corpus.ipynb)

# Part 3: Build the unlabeled pretraining corpus

By now we have cached skeleton sequences from notebook 02, one variable-length
`(T, 33, 3)` array per gait sequence, grouped by condition. This notebook turns
that pile into the two datasets the rest of the series needs: a large unlabeled
clip bank for pretraining, and a small labeled holdout for the final probe.

Three steps do the work. First we normalize each sequence so absolute camera
position and body size stop mattering, by centering on the pelvis and scaling by
torso length. Second we slice each normalized sequence into overlapping fixed
length windows, which multiplies a handful of long sequences into many short
training clips and gives the model a consistent input shape. Third we split: the
68 clinically labeled sequences become the probe holdout, and everything else,
labels or no labels, feeds pretraining.

That last point is the heart of the whole approach. Pretraining never looks at a
single label. It learns the shape of walking from the entire pool of clips, and
we save the precious labels for one tiny classifier at the very end.

## Run this locally or in Google Colab

In Colab, click the badge and run top to bottom. On your laptop, from the `gavd/`
folder, `uv sync` then `uv run jupyter lab 03-build-pretraining-corpus.ipynb`.

With `SMOKE_TEST = True`, the default, we synthesize a small set of labeled and
unlabeled sequences so you can watch the normalize, window, and split steps run
in seconds. With `SMOKE_TEST = False` we load the real per-condition skeleton
caches that notebook 02 wrote.

## Normalize, window, split

The figure shows the three transforms this notebook applies: pelvis-centering and
torso-scaling to remove the camera, sliding-window slicing to make many clips,
and the labeled-versus-unlabeled split that reserves the 68 clips for the probe.

![Normalize, window, split](images/corpus-build.svg)

*Normalize each sequence, slice it into overlapping windows, and split off the 68 labeled clips so pretraining stays label-free.*

## Colab setup

In [ ]:
# Colab setup and local .env loading.
import importlib.util, subprocess, sys

_import_name = {"scikit-learn": "sklearn", "opencv-python": "cv2",
                "yt-dlp": "yt_dlp", "python-dotenv": "dotenv"}

def _ensure(pkgs):
    """pip install any packages whose import is not already available."""
    missing = [p for p in pkgs if importlib.util.find_spec(_import_name.get(p, p)) is None]
    if missing:
        print("Installing:", " ".join(missing))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
    return missing

_ensure(["numpy", "pandas", "matplotlib", "python-dotenv", "tqdm"])
# This notebook only reshapes cached arrays, so it needs no video packages.

from dotenv import load_dotenv, find_dotenv
import os
load_dotenv(find_dotenv())
print("Loaded environment via load_dotenv(find_dotenv()).")

GAVD_CACHE_DIR = os.getenv("GAVD_CACHE_DIR")
print("Setup complete.")

## Configuration

In [ ]:
from pathlib import Path

CONFIG = {
    "SMOKE_TEST": False,         # True -> synthesize sequences. False -> load real caches.
    "CACHE_DIR": Path(GAVD_CACHE_DIR) if GAVD_CACHE_DIR else Path.cwd() / "cache",
    "T": 32,                     # frames per training window
    "STRIDE": 16,                # window stride (overlap = T - STRIDE)
    "C": 3,                      # channels per joint (x, y, z)
    "MIN_LEN": 16,               # skip sequences shorter than this many frames
    # The clinically labeled 5-class subset and the class order for labels.
    "LABELED_CLASSES": ["normal", "parkinsons", "stroke", "cerebral palsy", "myopathic"],
    "SEED": 42,
}
CONFIG["CACHE_DIR"].mkdir(parents=True, exist_ok=True)
print("CONFIG:")
for k, v in CONFIG.items():
    print(f"  {k:16s} = {v}")

## Constants and helpers

We reuse the same skeleton edges and semantic groups as every other notebook, and
we define the normalize function that removes the camera, a synthetic-skeleton
generator for smoke mode, and the inline animation helper so we can watch a
normalized clip before and after.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

EDGES = [
    (0,1),(1,2),(2,3),(0,4),(4,5),(5,6),(0,9),(0,10),(9,10),
    (11,12),(11,23),(12,24),(23,24),
    (11,13),(13,15),(15,17),(15,19),(15,21),(17,19),
    (12,14),(14,16),(16,18),(16,20),(16,22),(18,20),
    (23,25),(25,27),(27,29),(27,31),(29,31),
    (24,26),(26,28),(28,30),(28,32),(30,32),
]
GROUPS = {
    "face":[0,1,2,3,4,5,6,7,8,9,10], "left_arm":[11,13,15,17,19,21],
    "right_arm":[12,14,16,18,20,22], "torso":[11,12,23,24],
    "left_leg":[23,25,27,29,31], "right_leg":[24,26,28,30,32],
}

def normalize_skeleton_seq(seq):
    """Center on pelvis (mean of hips 23,24) and scale by torso length so camera
    position and scale do not matter. seq: (T, 33, 3) -> (T, 33, 3)."""
    seq = seq.astype(np.float32).copy()
    hips = (seq[:, 23, :] + seq[:, 24, :]) / 2.0
    shoulders = (seq[:, 11, :] + seq[:, 12, :]) / 2.0
    seq = seq - hips[:, None, :]
    torso = np.linalg.norm(shoulders - hips, axis=1)
    scale = np.median(torso[torso > 1e-6]) if np.any(torso > 1e-6) else 1.0
    return (seq / (scale + 1e-6)).astype(np.float32)

def synthesize_walking_skeleton(T=32, seed=0, gait_bias=0.0):
    """A plausible synthetic (T, 33, 3) walking skeleton for SMOKE mode."""
    rng = np.random.RandomState(seed)
    base = np.zeros((33, 3), dtype=np.float32)
    # Full (x, y) layout for all 33 landmarks, laid out as a person seen head-on.
    # y grows downward (head near 0.16, feet near 0.97). x has the midline at 0.50,
    # with the left side (odd joint indices) left of it and the right side right of it.
    # Shoulders are wider than the hips, the arms hang OUTSIDE the hips down to about
    # hip height, and the head sits just above the shoulders, so the figure reads as a
    # real walking body instead of collapsing onto one vertical line.
    xs = {
        0:0.500,                                            # nose
        1:0.485, 2:0.475, 3:0.465, 4:0.515, 5:0.525, 6:0.535,  # eyes (left then right)
        7:0.455, 8:0.545,                                   # ears
        9:0.485, 10:0.515,                                  # mouth
        11:0.415, 12:0.585,                                 # shoulders (wide)
        13:0.395, 14:0.605,                                 # elbows (arms hang outside)
        15:0.405, 16:0.595,                                 # wrists
        17:0.395, 18:0.605, 19:0.405, 20:0.595, 21:0.420, 22:0.580,  # hands track their wrist
        23:0.455, 24:0.545,                                 # hips (narrower than shoulders)
        25:0.450, 26:0.550,                                 # knees
        27:0.448, 28:0.552,                                 # ankles
        29:0.448, 30:0.552, 31:0.455, 32:0.545,             # heels, foot tips
    }
    ys = {
        0:0.16,                                             # nose
        1:0.145, 2:0.145, 3:0.145, 4:0.145, 5:0.145, 6:0.145,  # eyes
        7:0.155, 8:0.155,                                   # ears
        9:0.185, 10:0.185,                                  # mouth (short neck to shoulders)
        11:0.24, 12:0.24,                                   # shoulders
        13:0.38, 14:0.38,                                   # elbows
        15:0.51, 16:0.51,                                   # wrists (about hip height)
        17:0.545, 18:0.545, 19:0.545, 20:0.545, 21:0.535, 22:0.535,  # hands (just past wrists)
        23:0.50, 24:0.50,                                   # hips
        25:0.71, 26:0.71,                                   # knees
        27:0.92, 28:0.92,                                   # ankles
        29:0.94, 30:0.94, 31:0.965, 32:0.965,               # heels, foot tips
    }
    for j in range(33):
        base[j, 0] = xs[j]
        base[j, 1] = ys[j]
    seq = np.repeat(base[None], T, axis=0)
    t = np.linspace(0, 2*np.pi, T, endpoint=False)
    swing = 0.06 * np.sin(t)
    for k, amp in [(25,1.0),(27,1.3),(31,1.4),(13,-0.8),(15,-1.0)]:
        seq[:, k, 0] += swing * amp * (1.0 + gait_bias)
    for k, amp in [(26,-1.0),(28,-1.3),(32,-1.4),(14,0.8),(16,1.0)]:
        seq[:, k, 0] += swing * amp * (1.0 - gait_bias)
    seq += rng.randn(T, 33, 3).astype(np.float32) * 0.004
    return seq.astype(np.float32)

def animate_skeleton(seq, edges, title="Walking skeleton", fps=8):
    """Animate a (T, 33, C) skeleton inline (uses x=seq[...,0], y=seq[...,1])."""
    T = seq.shape[0]; x_all = seq[:, :, 0]; y_all = seq[:, :, 1]
    groups = [list(range(11)), [11,13,15,17,19,21], [12,14,16,18,20,22],
              [11,12,23,24], [23,25,27,29,31], [24,26,28,30,32]]
    colors = ["#8b5cf6","#3b82f6","#ef4444","#22c55e","#f59e0b","#ec4899"]
    fig, ax = plt.subplots(figsize=(6, 7))
    x_min, x_max = x_all.min(), x_all.max(); y_min, y_max = y_all.min(), y_all.max()
    margin = max(x_max - x_min, y_max - y_min) * 0.1 + 1e-3
    def draw_frame(t):
        ax.clear(); ax.set_aspect('equal'); ax.invert_yaxis(); ax.axis('off')
        ax.set_xlim(x_min - margin, x_max + margin); ax.set_ylim(y_max + margin, y_min - margin)
        ax.set_title(f"{title} (frame {t}/{T})")
        x = x_all[t]; y = y_all[t]
        for g_idx, grp in enumerate(groups):
            for (i, j) in [(i, j) for (i, j) in edges if i in grp and j in grp]:
                ax.plot([x[i], x[j]], [y[i], y[j]], color=colors[g_idx], linewidth=2, alpha=0.7)
            ax.scatter([x[i] for i in grp], [y[i] for i in grp], c=colors[g_idx],
                       s=40, zorder=3, edgecolors='white', linewidths=0.5)
    anim = FuncAnimation(fig, draw_frame, frames=T, interval=1000/fps, repeat=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())

print("Constants and helpers defined.")

## Load every cached sequence

We gather the skeleton sequences to process. In real mode we read each
`skeletons_<condition>.npz` file that notebook 02 wrote and pair every sequence
with its condition. In smoke mode we synthesize a handful of sequences per
condition, giving a few of the labeled classes a small left-right asymmetry so
the later probe has something to separate. Each entry is a tuple of condition,
sequence id, and its raw `(T, 33, 3)` array.

In [ ]:
import numpy as np

def load_real_sequences(cache_dir):
    seqs = []
    for npz in sorted(cache_dir.glob("skeletons_*.npz")):
        data = np.load(npz, allow_pickle=True)
        cond = str(data["condition"])
        for sid, arr in zip(data["seq_ids"], data["arrays"]):
            seqs.append((cond, str(sid), np.asarray(arr, dtype=np.float32)))
    return seqs

def synth_sequences():
    seqs = []
    plan = [("normal", 4, 0.0), ("parkinsons", 3, 0.25), ("stroke", 3, -0.3),
            ("cerebral palsy", 3, 0.15), ("myopathic", 3, 0.1),
            ("abnormal", 6, 0.05), ("style", 4, -0.05)]
    for cond, n, bias in plan:
        for i in range(n):
            T = 40 + 8 * (i % 3)
            arr = synthesize_walking_skeleton(T=T, seed=(hash((cond, i)) % 1000), gait_bias=bias)
            seqs.append((cond, f"cl{cond[:3]}{i:04d}synthetic0000000", arr))
    return seqs

if CONFIG["SMOKE_TEST"]:
    sequences = synth_sequences()
    print(f"SMOKE mode: synthesized {len(sequences)} sequences.")
else:
    sequences = load_real_sequences(CONFIG["CACHE_DIR"])
    if not sequences:
        report_path = CONFIG["CACHE_DIR"] / "extraction_report.csv"
        if report_path.exists():
            import pandas as pd
            extraction_report = pd.read_csv(report_path)
            ok_count = int(extraction_report["ok"].sum()) if "ok" in extraction_report else 0
            print(f"No skeleton caches found. Notebook 02 wrote extraction_report.csv with "
                  f"{ok_count}/{len(extraction_report)} successful sequences.")
            if "note" in extraction_report:
                print("Most common extraction notes:")
                print(extraction_report["note"].value_counts().head(5).to_string())
        print("No skeleton caches found; run notebook 02 first. Falling back to synthetic.")
        sequences = synth_sequences()
    else:
        print(f"Loaded {len(sequences)} real cached sequences.")

from collections import Counter
print("Per-condition sequence counts:", dict(Counter(c for c, _, _ in sequences)))

## See the effect of normalization

Normalization is what lets the model ignore where the camera was and how big the
person looked, and focus on how the body moved. The animation below plays one
normalized sequence: the pelvis sits at the origin and the body is scaled to a
consistent size, frame after frame. Every clip in the corpus goes through this
same transform.

In [ ]:
cond0, sid0, raw0 = sequences[0]
norm0 = normalize_skeleton_seq(raw0)
print(f"Animating a normalized '{cond0}' sequence ({norm0.shape[0]} frames).")
display(animate_skeleton(norm0[:24], EDGES, title=f"Normalized {cond0} skeleton"))

## Slice into overlapping windows

A single long sequence becomes many training clips when we slide a fixed-length
window across it with a stride. Overlap means neighboring windows share frames,
which multiplies the number of clips and helps the model see each moment in
different temporal contexts. We normalize first, then window, and we tag each
window with its condition and its source sequence id so we can trace it back.

In [ ]:
def window_sequence(seq, T, stride, min_len):
    """Normalize then slide a window of length T with the given stride.
    Returns a list of (T, 33, 3) clips. Short sequences yield one padded clip."""
    seq = normalize_skeleton_seq(seq)
    n = seq.shape[0]
    if n < min_len:
        return []
    if n < T:
        pad = np.repeat(seq[-1:], T - n, axis=0)      # repeat last frame to reach T
        return [np.concatenate([seq, pad], axis=0).astype(np.float32)]
    clips = []
    for start in range(0, n - T + 1, stride):
        clips.append(seq[start:start + T].astype(np.float32))
    return clips

all_clips, clip_condition, clip_seq = [], [], []
for cond, sid, raw in sequences:
    for clip in window_sequence(raw, CONFIG["T"], CONFIG["STRIDE"], CONFIG["MIN_LEN"]):
        all_clips.append(clip); clip_condition.append(cond); clip_seq.append(sid)

all_clips = np.stack(all_clips, axis=0) if all_clips else np.zeros((0, CONFIG["T"], 33, 3), np.float32)
print(f"Windowed {len(sequences)} sequences into {all_clips.shape[0]} clips "
      f"of shape {tuple(all_clips.shape[1:])}.")

## Split: unlabeled pretraining bank versus labeled holdout

Now we separate the clips. A clip is part of the labeled holdout only if its
condition is one of the five clinical classes and its source sequence was marked
as labeled in the manifest. In smoke mode we simply treat the five clinical
classes as labeled. Everything else, including abnormal, style, and any clinical
sequence beyond the labeled 68, goes into the unlabeled pretraining bank. The
model pretrains on the bank; the probe later uses only the holdout.

In [ ]:
import pandas as pd

labeled_seqs = set()
manifest_path = CONFIG["CACHE_DIR"] / "manifest.csv"
if not CONFIG["SMOKE_TEST"] and manifest_path.exists():
    man = pd.read_csv(manifest_path)
    labeled_seqs = set(man.loc[man.get("is_labeled", False) == True, "seq"].astype(str))
    print(f"Manifest marks {len(labeled_seqs)} sequences as labeled.")

labeled_classes = CONFIG["LABELED_CLASSES"]
class_to_idx = {c: i for i, c in enumerate(labeled_classes)}

def is_labeled_clip(cond, sid):
    if cond not in class_to_idx:
        return False
    if CONFIG["SMOKE_TEST"]:
        return True                     # smoke: all clinical-class clips count as labeled
    return sid in labeled_seqs          # real: only the manifest-marked 68 sequences

unl_clips, lab_clips, lab_labels, lab_seq = [], [], [], []
for i in range(all_clips.shape[0]):
    cond, sid = clip_condition[i], clip_seq[i]
    if is_labeled_clip(cond, sid):
        lab_clips.append(all_clips[i]); lab_labels.append(class_to_idx[cond]); lab_seq.append(sid)
    else:
        unl_clips.append(all_clips[i])

unl_clips = np.stack(unl_clips, 0) if unl_clips else np.zeros((0, CONFIG["T"], 33, 3), np.float32)
lab_clips = np.stack(lab_clips, 0) if lab_clips else np.zeros((0, CONFIG["T"], 33, 3), np.float32)
lab_labels = np.array(lab_labels, dtype=np.int64)

print(f"\nUnlabeled pretraining bank : {unl_clips.shape[0]} clips")
print(f"Labeled holdout            : {lab_clips.shape[0]} clips "
      f"from {len(set(lab_seq))} sequences across {len(set(lab_labels.tolist()))} classes")

## A picture of the split

The bar chart makes the imbalance concrete one more time: a tall unlabeled bank
that pretraining feeds on, and a short stack of labeled clips reserved for the
probe. This is the resource the whole method is designed to exploit.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(["unlabeled\n(pretraining)", "labeled\n(probe holdout)"],
       [unl_clips.shape[0], lab_clips.shape[0]], color=["#cbd5e1", "#ef4444"])
for i, v in enumerate([unl_clips.shape[0], lab_clips.shape[0]]):
    ax.text(i, v + 0.5, str(v), ha="center", fontsize=11)
ax.set_ylabel("number of clips")
ax.set_title("Pretraining bank vs labeled holdout")
plt.tight_layout(); plt.show()

## Save the corpus and the holdout

We write two files. `corpus.npz` holds the unlabeled clip bank the JEPA pretrains
on in notebook 04. `labeled_holdout.npz` holds the labeled clips, their integer
labels, their source sequence ids, and the class order, all of which notebook 05
needs to build and evaluate the frozen probe.

In [ ]:
corpus_path = CONFIG["CACHE_DIR"] / "corpus.npz"
np.savez(corpus_path, clips=unl_clips, T=CONFIG["T"], C=CONFIG["C"])

holdout_path = CONFIG["CACHE_DIR"] / "labeled_holdout.npz"
np.savez(holdout_path, clips=lab_clips, labels=lab_labels,
         seq_ids=np.array(lab_seq, dtype=object),
         classes=np.array(labeled_classes, dtype=object), T=CONFIG["T"], C=CONFIG["C"])

print(f"Wrote {corpus_path} ({unl_clips.shape[0]} unlabeled clips).")
print(f"Wrote {holdout_path} ({lab_clips.shape[0]} labeled clips).")

## Recap and what comes next

We normalized every cached sequence to erase the camera, sliced them into
overlapping fixed-length windows to multiply our training clips, and split the
result into a large unlabeled pretraining bank and a small labeled holdout. Only
the bank feeds pretraining; the holdout waits for the probe.

In notebook 04 we build the four JEPA pieces, the context encoder, the EMA target
encoder, the predictor, and the VICReg loss, and train them on the unlabeled bank
with block masking. We watch the loss fall while the anti-collapse terms stay
healthy, then save the trained encoder for the probe.